In [1]:
import os 
import cv2
import numpy as np

def add_fog_bgr(img_bgr, beta, A):
     
     """
        Takes one frame and return foggy version
        Fog model: I_fog = I*t + A*(1-t), with t=exp(-beta*d) **
        img_bgr= frame in OpenCV format
        beta= fog density
        A= how bright/grey fog veil
     """
     img = img_bgr.astype(np.float32) / 255.0 # convert frame to float 0-1, easier to do math in normalized floats
     H,W = img.shape[:2] # height/width of image
    
    # create a grid where every pixel knows dist from vcenter of image
     y = np.arange(H,dtype=np.float32)[:,None]
     x= np.arange(W,dtype=np.float32)[None,:]
     cy,cx = H / 2.0, W/2.0
     dist = np.sqrt((y-cy)**2 + (x-cx)**2)

     # compute fake depth
     size = np.sqrt(max(H,W))
     d = -0.04 * dist + size # value goes down towards corners & shifts everything positive
     # d -> bigger = heavier fog effect

     # compute transmission -> how much from og image survives through the fog
     t= np.exp(-beta * d)
     t = np.clip(t,0.0,1.0)
     # t smaller when fog is strong -> image gets washed out
     # beta controls strenght of fog

     # mix frame with fog veil
     # foggy_pixel = og_pixel * t + fog_color * (1-t)
     # t = 1 -> og image / t = 0 -> fog veil
     # t[...] apply to all three RGB
     fog = img * t[..., None] + A * (1.0 - t[..., None])
     return np.clip(fog*255.0,0,255).astype(np.uint8)

In [2]:
def fog_video(in_path, out_path, beta, A):
    cap = cv2.VideoCapture(in_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {in_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # mp4v is usually fine; if writer issues, switch to XVID + .avi
    out_path = os.path.splitext(out_path)[0] + ".avi"
    fourcc = cv2.VideoWriter_fourcc(*"XVID")
    writer = cv2.VideoWriter(out_path, fourcc, fps, (w, h))
    if not writer.isOpened():
        raise RuntimeError("Writer failed to open")  
    n = 0
    while True:
        ok, frame = cap.read()

        # HARD GUARD: never process invalid frames
        if not ok or frame is None:
            print(f"Stopped reading {os.path.basename(in_path)} at frame {n}. ok={ok}, frame_is_none={frame is None}")
            break

        foggy = add_fog_bgr(frame, beta=beta, A=A)
        writer.write(foggy)
        n += 1

        if n % 300 == 0:
            print(f"  {os.path.basename(in_path)}: processed {n} frames...")

    cap.release()
    writer.release()

    if n == 0:
        # This should basically never happen now, but if it does, we’ll know.
        print("WARNING: wrote 0 frames for", in_path)
    else:
        print(f"Saved: {out_path} ({n} frames)")

In [3]:
def process_split(split_dir, beta, A, visible_name="visible.mp4" ):
    """
    split_dir: e.g. /path/to/dataset/train
    It will find every visible.mp4 in subfolders and generate a foggy version next to it.
    """
    if not os.path.isdir(split_dir):
        raise RuntimeError(f"Split dir not found: {split_dir}")

    print("Processing split:", split_dir)
    count = 0

    # walk all subfolders
    for root, _, files in os.walk(split_dir):
        if visible_name in files:
            in_vid = os.path.join(root, visible_name)
            out_vid = os.path.join(root, f"visible_fog_beta{beta:.2f}.mp4")

            if os.path.exists(out_vid):
                print("Skipping (already exists):", out_vid)
                continue

            print("Fogging:", in_vid)
            fog_video(in_vid, out_vid, beta=beta, A=A)
            count += 1

    print(f"Done. Processed {count} videos in {split_dir}")

In [4]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
def main():
     dataset_root = "/content/drive/MyDrive/thesis-1/dataset/Anti-UAV-RGBT/"
     #visible_name = "visible.mp4"
     
     beta = 0.07  
     A = 0.5
     
     split = "train" # change to test or val
     split_dir = f"{dataset_root}/train"

     process_split(split_dir, visible_name="visible.mp4", beta=beta ,A=A)

if __name__  == "__main__":
    main()

Processing split: /content/drive/MyDrive/thesis-1/dataset/Anti-UAV-RGBT//train
Fogging: /content/drive/MyDrive/thesis-1/dataset/Anti-UAV-RGBT//train/20190926_200510_1_5/visible.mp4
  visible.mp4: processed 300 frames...
  visible.mp4: processed 600 frames...
  visible.mp4: processed 900 frames...
Stopped reading visible.mp4 at frame 1000. ok=False, frame_is_none=True
Saved: /content/drive/MyDrive/thesis-1/dataset/Anti-UAV-RGBT//train/20190926_200510_1_5/visible_fog_beta0.07.avi (1000 frames)
Fogging: /content/drive/MyDrive/thesis-1/dataset/Anti-UAV-RGBT//train/20190926_200510_1_7/visible.mp4
  visible.mp4: processed 300 frames...


KeyboardInterrupt: 

In [13]:
import os
print("MyDrive exists?", os.path.exists("/content/drive/MyDrive"))
print("Top of MyDrive:", os.listdir("/content/drive/MyDrive")[:50])

MyDrive exists? True
Top of MyDrive: ['autorizacion.docx', 'autorizacion.gdoc', 'Archivo_000.png', 'timeteable.gsheet', 'PRODUCT FLIPING BUSINESS.gsheet', 'Viaje gur.pdf', 'REALTOR SHIT.zip', 'Takeout', 'security protocol summary 113.MOV', 'File_000', 'פטור גור.jpg', 'פטור גור.gdoc', 'EANE_SOLICITUD_INSCRIPCION_AGENTE_INMOBILIARIO.docx', 'DSCF0964.JPG', 'DSCF0915.JPG', 'DSCF0906.JPG', 'DSCF0959.JPG', 'DSCF0971.JPG', 'DSCF0923.JPG', 'DSCF0901.JPG', 'DSCF0957.JPG', 'DSCF0928.JPG', 'DSCF0914.JPG', 'DSCF0948.JPG', 'DSCF0975.JPG', 'DSCF0908.JPG', 'DSCF0960.JPG', 'DSCF0904.JPG', 'DSCF0927.JPG', 'DSCF0930.JPG', 'DSCF0902.JPG', 'DSCF0903.JPG', 'DSCF0961.JPG', 'DSCF0946.JPG', 'DSCF0912.JPG', 'DSCF0970.JPG', 'DSCF0954.JPG', 'DSCF0944.JPG', 'DSCF0942.JPG', 'DSCF0955.JPG', 'DSCF0919.JPG', 'DSCF0967.JPG', 'DSCF0916.JPG', 'DSCF0972.JPG', 'DSCF0945.JPG', 'DSCF0963.JPG', 'DSCF0965.JPG', 'DSCF0941.JPG', 'DSCF0925.JPG', 'DSCF0949.JPG']


In [2]:
import os

target = "datasets"
root = "/content/drive/MyDrive"

hits = []
for dirpath, dirnames, filenames in os.walk(root):
    if target in dirnames:
        hits.append(os.path.join(dirpath, target))

print("Found:", hits[:20])
print("Total:", len(hits))

Found: ['/content/drive/MyDrive/datasets', '/content/drive/MyDrive/datasets/Anti-UAV/anti_uav_jittor/pysot_toolkit/toolkit/datasets', '/content/drive/MyDrive/datasets/Anti-UAV/anti_uav_jittor/got10k_toolkit/toolkit/datasets', '/content/drive/MyDrive/datasets/Anti-UAV/anti_uav_jittor/anti_uav410_jit/datasets', '/content/drive/MyDrive/datasets/Anti-UAV/anti_uav_jittor/anti_uav410_jit/trackers/SiamDT/datasets', '/content/drive/MyDrive/datasets/Anti-UAV/anti_uav_jittor/anti_uav410_jit/trackers/SiamDT/libs/swintransformer/mmdet/datasets', '/content/drive/MyDrive/datasets/Anti-UAV/anti_uav_jittor/anti_uav410_jit/trackers/SiamDT/libs/swintransformer/configs/_base_/datasets', '/content/drive/MyDrive/datasets/Anti-UAV/anti_uav_jittor/anti_uav410_jit/trackers/SiamDT/libs/data/datasets']
Total: 8
